# Filing Momentum ML Equity Curve

This notebook plots the Filing Momentum ML strategy equity curve against the S&P 500 proxy used by the backtest benchmark (`SPY`). It reads cached quarterly curve data from `outputs/filing_momentum_ml_equity_curve.csv` when available; set `REFRESH_BACKTEST = True` below to recompute it from the local raw data and manifest. The chart uses a business-day interpolated plotting view between quarter-end backtest observations so the line is easier to read; the quarter-end markers remain the source-of-truth observations.

In [ ]:
from pathlib import Path
import sys


def resolve_project_root() -> Path:
    candidates = []
    try:
        candidates.append(Path.cwd())
    except FileNotFoundError:
        pass
    candidates.append(Path("/Users/gavinhussey/Downloads/AtlasQuant"))
    for candidate in candidates:
        if candidate.name == "outputs":
            candidate = candidate.parent
        if (candidate / "src" / "atlas_quant").exists():
            return candidate
    raise FileNotFoundError("Could not locate AtlasQuant project root")


PROJECT_ROOT = resolve_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

OUTPUT_DIR = PROJECT_ROOT / "outputs"
CURVE_CSV = OUTPUT_DIR / "filing_momentum_ml_equity_curve.csv"
RAW_ROOT = PROJECT_ROOT / "data" / "raw" / "filing_momentum_ml"
MANIFEST = PROJECT_ROOT / "data" / "manifests" / "filing_momentum_ml" / "data_manifest.json"
CHECKPOINT_ROOT = PROJECT_ROOT / "data" / "manifests" / "filing_momentum_ml"

START_QUARTER = "2015-03-31"
END_QUARTER = "2026-06-30"
BENCHMARK_SYMBOL = "SPY"
REFRESH_BACKTEST = False
PLOT_FREQUENCY = "B"  # business-day interpolation for chart readability

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("project root:", PROJECT_ROOT)
print("curve cache:", CURVE_CSV)

In [ ]:
import numpy as np
import pandas as pd


def run_backtest_equity_curve() -> pd.DataFrame:
    from argparse import Namespace

    from atlas_quant.cli.filing_momentum import (
        _build_run_inputs,
        _build_trading_calendar,
        _periods_from_args,
        load_normalized_bundle,
    )
    from atlas_quant.strategies.filing_momentum_ml.production.orchestration import (
        ProductionRunState,
        run_filing_momentum_production_backtest,
    )

    args = Namespace(
        raw_root=RAW_ROOT,
        manifest=MANIFEST,
        start_quarter=START_QUARTER,
        end_quarter=END_QUARTER,
        earnings_lag_days=None,
        benchmark=BENCHMARK_SYMBOL,
        checkpoint_root=CHECKPOINT_ROOT,
        dry_run=False,
    )
    bundle = load_normalized_bundle(args.raw_root)
    calendar = _build_trading_calendar(bundle)
    periods = _periods_from_args(args, required=True)
    inputs = _build_run_inputs(args, bundle, calendar, periods, with_report=False)
    result = run_filing_momentum_production_backtest(inputs)
    if result.state not in (ProductionRunState.COMPLETED, ProductionRunState.COMPLETED_WITH_WARNINGS):
        missing = ", ".join(d.name for d in result.missing_dependencies) or "none"
        raise RuntimeError(f"backtest did not complete: {result.state.value}; {result.blocked_reason}; missing={missing}")
    if result.backtest_result is None:
        raise RuntimeError("backtest completed without a BacktestResult")

    rows = [
        {
            "quarter_end": quarter_end.isoformat(),
            "filing_momentum_ml_equity": strategy_equity,
            "sp500_equity": benchmark_equity,
        }
        for quarter_end, strategy_equity, benchmark_equity in result.backtest_result.equity_curve()
    ]
    df = pd.DataFrame(rows)
    df.to_csv(CURVE_CSV, index=False)
    return df


if CURVE_CSV.exists() and not REFRESH_BACKTEST:
    curve = pd.read_csv(CURVE_CSV)
else:
    curve = run_backtest_equity_curve()

curve["quarter_end"] = pd.to_datetime(curve["quarter_end"])
curve = curve.sort_values("quarter_end").reset_index(drop=True)


def build_plot_curve(df: pd.DataFrame, *, frequency: str = PLOT_FREQUENCY) -> pd.DataFrame:
    """Create a dense display-only equity curve from quarterly anchors.

    The backtest emits quarter-end observations. For chart readability,
    interpolate log equity over business days while preserving each
    quarter-end value exactly.
    """
    if df.empty:
        return df.copy()
    indexed = df.set_index("quarter_end").sort_index()
    numeric_cols = ["filing_momentum_ml_equity", "sp500_equity"]
    date_index = pd.date_range(indexed.index.min(), indexed.index.max(), freq=frequency)
    date_index = date_index.union(indexed.index).sort_values()
    log_dense = np.log(indexed[numeric_cols]).reindex(date_index).interpolate(method="time")
    dense = np.exp(log_dense)
    dense.index.name = "quarter_end"
    return dense.reset_index()


plot_curve = build_plot_curve(curve)
print(f"quarter-end observations: {len(curve)}; plotted points: {len(plot_curve)}")
curve.tail()

In [ ]:
from html import escape
from IPython.display import HTML, display


def render_equity_curve_svg(df, *, anchor_df=None, width=980, height=540):
    if df.empty:
        return "<p>No equity-curve data available.</p>"

    margin = {"top": 46, "right": 34, "bottom": 58, "left": 72}
    plot_w = width - margin["left"] - margin["right"]
    plot_h = height - margin["top"] - margin["bottom"]
    values = list(df["filing_momentum_ml_equity"]) + list(df["sp500_equity"])
    y_min = min(1.0, min(values))
    y_max = max(1.0, max(values))
    padding = (y_max - y_min) * 0.08 or 0.1
    y_min -= padding
    y_max += padding

    start = df["quarter_end"].min()
    end = df["quarter_end"].max()
    span_days = max((end - start).days, 1)

    def x_at_date(dt):
        return margin["left"] + ((dt - start).days / span_days) * plot_w

    def y_at(v):
        return margin["top"] + (y_max - v) / (y_max - y_min) * plot_h

    def polyline(column):
        return " ".join(
            f"{x_at_date(row.quarter_end):.1f},{y_at(getattr(row, column)):.1f}"
            for row in df.itertuples(index=False)
        )

    def markers(column, color):
        if anchor_df is None or anchor_df.empty:
            return ""
        return "".join(
            f'<circle cx="{x_at_date(row.quarter_end):.1f}" cy="{y_at(getattr(row, column)):.1f}" r="2.6" fill="#ffffff" stroke="{color}" stroke-width="1.6"/>'
            for row in anchor_df.itertuples(index=False)
        )

    grid = []
    labels = []
    for j in range(6):
        value = y_min + (y_max - y_min) * j / 5
        y = y_at(value)
        grid.append(f'<line x1="{margin["left"]}" y1="{y:.1f}" x2="{width - margin["right"]}" y2="{y:.1f}" stroke="#e5e7eb"/>')
        labels.append(f'<text x="{margin["left"] - 12}" y="{y + 4:.1f}" text-anchor="end" font-size="12" fill="#4b5563">{value:.2f}x</text>')

    year_ticks = pd.date_range(start=start, end=end, freq="YS")
    if len(year_ticks) == 0 or year_ticks[0] != start:
        year_ticks = year_ticks.union(pd.DatetimeIndex([start]))
    if year_ticks[-1] != end:
        year_ticks = year_ticks.union(pd.DatetimeIndex([end]))
    x_labels = []
    for tick in year_ticks:
        x = x_at_date(tick)
        label = tick.strftime("%Y")
        x_labels.append(f'<text x="{x:.1f}" y="{height - 24}" text-anchor="middle" font-size="12" fill="#4b5563">{escape(label)}</text>')

    strategy_final = df["filing_momentum_ml_equity"].iloc[-1]
    sp500_final = df["sp500_equity"].iloc[-1]
    svg = f'''
    <svg viewBox="0 0 {width} {height}" width="100%" height="{height}" role="img" aria-label="Filing Momentum ML and S&P 500 equity curves">
      <rect width="{width}" height="{height}" fill="#ffffff"/>
      <text x="{margin['left']}" y="24" font-size="20" font-weight="700" fill="#111827">Filing Momentum ML vs S&amp;P 500 Equity Curve</text>
      <text x="{margin['left']}" y="42" font-size="12" fill="#6b7280">Business-day interpolated display; circles mark quarter-end backtest observations.</text>
      {''.join(grid)}
      {''.join(labels)}
      <line x1="{margin['left']}" y1="{height - margin['bottom']}" x2="{width - margin['right']}" y2="{height - margin['bottom']}" stroke="#9ca3af"/>
      <line x1="{margin['left']}" y1="{margin['top']}" x2="{margin['left']}" y2="{height - margin['bottom']}" stroke="#9ca3af"/>
      {''.join(x_labels)}
      <polyline points="{polyline('sp500_equity')}" fill="none" stroke="#6b7280" stroke-width="2.4" stroke-linejoin="round" stroke-linecap="round"/>
      <polyline points="{polyline('filing_momentum_ml_equity')}" fill="none" stroke="#2563eb" stroke-width="3" stroke-linejoin="round" stroke-linecap="round"/>
      {markers('sp500_equity', '#6b7280')}
      {markers('filing_momentum_ml_equity', '#2563eb')}
      <rect x="{width - 315}" y="54" width="250" height="76" rx="6" fill="#ffffff" stroke="#d1d5db"/>
      <line x1="{width - 295}" y1="82" x2="{width - 255}" y2="82" stroke="#2563eb" stroke-width="4"/>
      <text x="{width - 245}" y="86" font-size="13" fill="#111827">Filing Momentum ML: {strategy_final:.2f}x</text>
      <line x1="{width - 295}" y1="110" x2="{width - 255}" y2="110" stroke="#6b7280" stroke-width="4"/>
      <text x="{width - 245}" y="114" font-size="13" fill="#111827">S&amp;P 500 (SPY): {sp500_final:.2f}x</text>
    </svg>
    '''
    return svg


display(HTML(render_equity_curve_svg(plot_curve, anchor_df=curve)))

In [ ]:
summary = pd.DataFrame(
    [
        {"series": "Filing Momentum ML", "final_equity": curve["filing_momentum_ml_equity"].iloc[-1], "total_return": curve["filing_momentum_ml_equity"].iloc[-1] - 1},
        {"series": "S&P 500 (SPY)", "final_equity": curve["sp500_equity"].iloc[-1], "total_return": curve["sp500_equity"].iloc[-1] - 1},
    ]
)
summary

## Strategy statistics

Sharpe and Sortino reuse `atlas_quant`'s own tested `compute_sharpe`/`compute_sortino`
(report §6/§7 formulas, quarterly-periods-per-year annualized, 0% risk-free rate,
population stddev -- this platform's established convention) rather than
re-deriving the math here. Max drawdown is the absolute compounded-equity
peak-to-trough decline at quarterly resolution (same definition as
`compute_drawdown`), computed directly off this notebook's equity curve.

In [ ]:
from atlas_quant.strategies.filing_momentum_ml.performance_domain import PerformanceAnalysisConfig
from atlas_quant.strategies.filing_momentum_ml.performance_metrics import compute_sharpe, compute_sortino

PERF_CONFIG = PerformanceAnalysisConfig()


def quarterly_returns(equity: pd.Series) -> list[float]:
    return (equity / equity.shift(1) - 1).dropna().tolist()


def max_drawdown_peak_to_trough(quarter_end: pd.Series, equity: pd.Series) -> dict:
    """Absolute compounded-equity peak-to-trough decline -- same definition as
    atlas_quant.strategies.filing_momentum_ml.performance_metrics.compute_drawdown,
    applied directly to this notebook's already-compounded equity curve."""
    running_max = equity.iloc[0]
    peak_date = quarter_end.iloc[0]
    best = {"max_drawdown": 0.0, "peak_date": peak_date, "trough_date": peak_date}
    for qend, eq in zip(quarter_end, equity):
        if eq > running_max:
            running_max = eq
            peak_date = qend
        dd = eq / running_max - 1
        if dd < best["max_drawdown"]:
            best = {"max_drawdown": dd, "peak_date": peak_date, "trough_date": qend}
    return best


def strategy_stats(label: str, equity: pd.Series, quarter_end: pd.Series) -> dict:
    returns = quarterly_returns(equity)
    sharpe = compute_sharpe(returns, PERF_CONFIG)
    sortino = compute_sortino(returns, PERF_CONFIG)
    dd = max_drawdown_peak_to_trough(quarter_end, equity)
    total_return = equity.iloc[-1] / equity.iloc[0] - 1
    n_years = (quarter_end.iloc[-1] - quarter_end.iloc[0]).days / 365.25
    cagr = (equity.iloc[-1] / equity.iloc[0]) ** (1 / n_years) - 1 if n_years > 0 else None
    return {
        "series": label,
        "total_return": total_return,
        "cagr": cagr,
        "sharpe": sharpe.value,
        "sortino": sortino.value,
        "max_drawdown": dd["max_drawdown"],
        "max_dd_peak": dd["peak_date"].date(),
        "max_dd_trough": dd["trough_date"].date(),
        "best_quarter": max(returns) if returns else None,
        "worst_quarter": min(returns) if returns else None,
        "win_rate": (sum(1 for r in returns if r > 0) / len(returns)) if returns else None,
        "n_quarters": len(returns),
    }


stats = pd.DataFrame([
    strategy_stats("Filing Momentum ML", curve["filing_momentum_ml_equity"], curve["quarter_end"]),
    strategy_stats("S&P 500 (SPY)", curve["sp500_equity"], curve["quarter_end"]),
]).set_index("series")

signed_pct_cols = ["total_return", "cagr", "max_drawdown", "best_quarter", "worst_quarter"]
formatted = stats.assign(**{c: stats[c].map(lambda v: f"{v:+.2%}" if pd.notna(v) else "n/a") for c in signed_pct_cols})
formatted = formatted.assign(
    win_rate=stats["win_rate"].map(lambda v: f"{v:.1%}" if pd.notna(v) else "n/a"),
    sharpe=stats["sharpe"].map(lambda v: f"{v:.2f}" if pd.notna(v) else "n/a"),
    sortino=stats["sortino"].map(lambda v: f"{v:.2f}" if pd.notna(v) else "n/a"),
)
display(formatted)